# Text Similarity (STS) – Overlap similarity

## Project Overview

This notebook is part of a research project devoted to the **estimation of semantic textual similarity (STS)** using lexical-semantic resources and vector-based representations. The objective is to implement and compare several approaches for computing sentence-level semantic similarity by **combining standard lexical similarity measures with different aggregation strategies at the sentence level**. Four alternative aggregation strategies are explored in this project. Each approach is implemented in a different notebook:

1) **[Notebook 1](PLN-Proyecto%20-%20PARTE1%20-%20Liw-Wang%20con%20funciones%20wordnet.ipynb): Cosine similarity between semantic vectors derived from lexical similarity scores** as described in [Liu & Wang 2014](https://www.jsoftware.us/vol9/jsw0902-32.pdf)
2) **[Notebook 2](PLN-Proyecto%20-%20PARTE2%20-%20Ariadna%20con%20funciones%20wordnet.ipynb): A pairwise synset similarity aggregation strategy** (proposed in this project)
3) **Notebook 3: Overlap similarity**  (this notebook, based on the approach described in [Agirre et al. 2017](https://aclanthology.org/S17-2001/))

All notebooks in this project share the same experimental pipeline. The workflow includes **text preprocessing, synset assignment using WordNet (when applicable), computation of lexical similarity between tokens, and aggregation of these similarities into sentence-level similarity scores**. The notebooks differ in the formulation of the **sentence similarity stage**, while the preceding components remain identical in order to ensure comparability across experiments.

The evaluation is performed using sentence pairs from the **STSBenchmark** dataset, which provides human similarity judgments for pairs of English sentences. System scores produced by the implemented methods are compared with the gold-standard annotations using **Pearson correlation**.

Intermediate results and similarity scores generated by the notebooks were exported and analyzed externally. A summary of the experimental results is available in the [project report](../Procedimiento%20y%20discusión.pdf). Pearson computations can be consulted in the [experiments comparison results file](../comparativa.xlsx).

## Subproject Overview – Overlap similarity

This notebook implements a sentence similarity method based on **overlap between the lexical units present in two sentences**, following the approach proposed by Agirre et al. The method estimates semantic similarity by measuring the **proportion of shared elements between the two sentences**.

Two variants of the overlap computation are explored. The first follows the original formulation and computes similarity using the **tokens extracted from the sentences**. The second variant replaces tokens with their corresponding **WordNet synsets**, allowing overlap to be measured at the **concept level rather than the surface word level**.

Sentence similarity is computed as the **normalized overlap between the two sets of elements**, producing a score that reflects the degree of shared lexical or semantic content between the sentences.

## Implementation

### Experiment configuration, dependencies, and auxiliary functions

In the following cells we set the **global configuration of the experiment** and load the required libraries. The main experimental parameters controlling the execution of the notebook are defined here. These include whether **Lesk-based word sense disambiguation** is applied, the type of **bag of words / bag of synsets** considered in the calculations, the **sentence-level similarity method** to be used, and the **corpus partition** to be processed.

Additional Boolean variables control whether libraries and data should be imported during execution. A descriptive string (`criterios`) is also created to identify the configuration of each experimental run when results are exported to an Excel file.

In [ ]:
# Parámetros a definir

    # lesk: True/False
        # True --> Synset se elige utilizando el desambiguador de Lesk
        # False --> Synset es el más común 
    # tipo_bow: --> Categorías de synsets que cogeremos para hacer los cálculos
        # todos (1) --> Todos los synsets
        # nv (2) --> Solo nombres y adjetivos
    # metodo_frase: lw / Ariadna / Agirre
    # partición: numero entero
        # fragmento del corpus que se procesa.
        

desambiguar = True

expandir_contracciones = False

tipo_bow = 'nv'

metodo_frase = 'Agirre'

particion = 1

print_info = False

importar_librerias = True

importar_datos = True

criterios = 'Lesk-' + str(desambiguar) + ', tipo_bow-' + str(tipo_bow) + ', metodo-' + str(metodo_frase) + ', particion-' + str(particion)

print()
print('Desambiguación desambiguar:', desambiguar)
print('Tipo de BoW:', tipo_bow)
print('Método de comparación a nivel de frase:', metodo_frase)
print('Particion de filas a importar:', particion)
print('Importar librerías:', importar_librerias)
print('Importar datos:', importar_datos)

print(criterios)



In [ ]:
# Importación de librerías

if importar_librerias:
    
    import time
    import datetime

    import csv

    import pandas as pd
    pd.options.display.max_rows  # Para mostrar todas las columnas
    pd.set_option('display.max_colwidth', None)  # Para incluir todo el contenido de una celda, sin truncar contenido.
    pd.set_option('display.max_columns', 500)  # Para incluir todas las columnas (no serán más de 500)
    pd.set_option('display.max_rows', 6000)  # Para incluir todas las filas (no serán más de 6000)
    
    import nltk

    def ensure_nltk_resource(resource_path, download_name):
        try:
            nltk.data.find(resource_path)
        except LookupError:
            nltk.download(download_name)

    # Recursos necesarios de NLTK
    ensure_nltk_resource("corpora/wordnet", "wordnet")
    ensure_nltk_resource("corpora/omw-1.4", "omw-1.4")
    ensure_nltk_resource("corpora/wordnet_ic", "wordnet_ic")
    ensure_nltk_resource("corpora/genesis", "genesis")

    from nltk.corpus import wordnet as wn

    from nltk.corpus import wordnet_ic
    ic_brown = wordnet_ic.ic('ic-brown.dat')
    ic_semcor = wordnet_ic.ic('ic-semcor.dat')

    from nltk.corpus import genesis
    ic_genesis = wn.ic(genesis, False, 0.0)
    
    if desambiguar == True:
        from nltk.wsd import lesk

import numpy as np

### Dataset import, cleaning and reduced corpus creation

The **STSBenchmark training dataset** is loaded into a pandas dataframe. The file contains pairs of English sentences annotated with **human similarity scores**, which serve as the reference values for evaluation. Duplicated sentence pairs are removed to avoid counting identical examples more than once during the experiments.

A reduced dataframe (`c`) is then created by selecting rows from the original corpus at regular intervals. This subset is used during development to **test preprocessing and similarity algorithms more quickly**, allowing faster iteration before running the full experiment on the complete dataset.

In [ ]:
# Importación de los datos del fichero sts-train.csv a un dataframe

start_total = time.time()

if importar_datos:

    file = '../stsbenchmark/sts-train.csv'
    corpus = pd.read_csv(file, sep='\t', on_bad_lines='skip', quoting=csv.QUOTE_NONE, usecols=range(7), header=None)
    corpus.head()
    corpus.columns = ['genre', 'file', 'year', 'id', 'gold_score', 'sent1', 'sent2']


end_total = time.time()

print('TIEMPO TRANSCURRIDO:', end_total-start_total)

corpus.sample(5)

In [ ]:
# Eliminación de duplicados

if importar_datos:

    corpus.drop_duplicates(['sent1', 'sent2'], keep='first', inplace=True)

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

In [ ]:
# Creación de un corpus c reducido con filas representativas del corpus original

    # Para realizar pre-procesamientos que nos permitan verificar la adecuación de los algoritmos
    # en un tiempo reducido.

particion = particion

c = corpus.iloc[::particion, :].copy()
num_filas = 'Número de filas incluídas en el dataframe: ' + str(c.shape[0])

print(num_filas)

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

c.head(12)

### Text preprocessing

Each sentence in the dataset undergoes the following standard natural language processing steps:  
1) tokenization  
2) lowercase conversion  
3) part-of-speech tagging*  
4) stopword removal (to retain only content-bearing words).

The resulting tokens are then prepared for the subsequent stages of the pipeline. The preprocessing results are stored in additional columns of the dataframe so that both the original sentences and their processed representations remain available.

A method for expanding contractions is implemented but not currently used (kept for potential future improvements).

In [ ]:
# ========================================
# 3. TOKENIZACION, LIMPIEZA Y ANOTACIÓN
# ========================================

def expandir_contracciones_texto(texto):
    # specific
    texto = re.sub(r"won\'t", "will not", texto)
    texto = re.sub(r"can\'t", "can not", texto)

    # general
    texto = re.sub(r"n\'t", " not", texto)
    texto = re.sub(r"\'re", " are", texto)
    texto = re.sub(r"\'s", " is", texto)
    texto = re.sub(r"\'d", " would", texto)
    texto = re.sub(r"\'ll", " will", texto)
    texto = re.sub(r"\'t", " not", texto)
    texto = re.sub(r"\'ve", " have", texto)
    texto = re.sub(r"\'m", " am", texto)
    return texto

def tokenizar(sent, print_tokenizar =False):
    
    # tokens que aceptaremos (de nltk.org/book/ch03). Output: cadena de palabras.
   
    if print_tokenizar == True: print('Frase a tokenizar:', sent)
        
    pattern = r'''(?x)       # set flag to allow verbose regexps
        (?:[A-Z]\.)+         # abbreviations, e.g. U.S.A. ?: needs to be added to specify that the parenthesis specify the scope of the disjunction, not the selection o strings to be extracted
       | \w+(?:-\w+)*        # words with optional internal hyphens
       | [\$,\€]?\d+(?:\.\d+)?%?  # currency and percentages, e.g. $12.40, 82%
    '''
    if print_tokenizar == True: print('pattern:', pattern)
    '''
    AMPLIAR.
    - Analizar errores
    - Alguna solución para detectar phrasal verbs?
    '''

    # tokenización. Output: lista
   
    tokens = nltk.regexp_tokenize(sent, pattern, gaps=False)
    
    # limpieza 1: mayúsculas
    
    tokens = [token.lower() for token in tokens]   
    if print_tokenizar == True: print('Tokens:', tokens)
 
    # anotación POS. Output: lista de tuplas
    
    tagged_tokens = nltk.pos_tag(tokens, tagset='universal')
    
    # Convertimos las etiquetas gramaticales en notación Lesk;
        
    for i, tagged_token in enumerate(tagged_tokens):
        if tagged_tokens[i][1] == 'NOUN': tagged_tokens[i] = (tagged_tokens[i][0], 'n')
        if tagged_tokens[i][1] == 'VERB': tagged_tokens[i] = (tagged_tokens[i][0], 'v')
        if tagged_tokens[i][1] == 'ADJ': tagged_tokens[i] = (tagged_tokens[i][0], 'a')
        if tagged_tokens[i][1] == 'ADV': tagged_tokens[i] = (tagged_tokens[i][0], 'r')
    if print_tokenizar == True: print('Tagged_tokens:', tagged_tokens)
    
    # limipieza 2: stopwords - REVISAR si queremos/podemos incluir phrasal verbs, determinantes, pronombres...
            
                # Wordnet no incluye determinantes o preposiciones, de forma que "a" es asociado por Lesk 
                # al Synset('deoxyadenosine_monophosphate.n.01'), lo cual no nos conviene.
                # Por otro lado, Lesk utiliza para desambiguar solo las palabras que aparecen en Wordnet,
                # de modo que limpiar antes o después de Lesk las stopwords no debería afectar al resultado.
   
    if print_tokenizar == True: print('Limpieza de stopwords')
    stopwords = nltk.corpus.stopwords.words('english')
    tagged_stops = [tagged_stop for tagged_stop in tagged_tokens if tagged_stop[0] in stopwords]
    if print_tokenizar == True: print('Stopwords eliminadas:', tagged_stops)
    tokens = [token for token in tokens if token not in stopwords]
    tagged_tokens = [tagged_token for tagged_token in tagged_tokens if tagged_token[0] not in stopwords]
    if print_tokenizar == True: print('Tagged tokens sin stopwords:', tagged_tokens)

    return(tokens, tagged_tokens, tagged_stops)

# Ejemplo para comprobar funcionamiento tokenizar()
   
sent = "I'll always love my sweet baby"

print('****** EJEMPLO tokenizar() ********')
tokens = tokenizar(sent, print_tokenizar=True)
print('****')

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)


In [ ]:
# Tokenización de c

c['tokens1'], c['tokens2'] = None, None
c['tagged_tokens1'], c['tagged_tokens2'] = None, None
c['stops1'], c['stops2'] = None, None

print_tokenizar = False

for i in c.index:
    tok1 = tokenizar(c.at[i, 'sent1'], print_tokenizar)
    tok2 = tokenizar(c.at[i, 'sent2'], print_tokenizar)

    c.at[i, 'tokens1'] = tok1[0]
    c.at[i, 'tokens2'] = tok2[0]
    c.at[i, 'tagged_tokens1'] = tok1[1]
    c.at[i, 'tagged_tokens2'] = tok2[1]
    c.at[i, 'stops1'] = tok1[2]
    c.at[i, 'stops2'] = tok2[2]

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

c.head()

### Synset extraction

Preprocessed tokens are associated with **WordNet synsets** when possible. When enabled, **Lesk-based word sense disambiguation** is applied to select the most appropriate synset for each token; otherwise the first synset provided by WordNet is used. Tokens for which no synset is found are recorded separately.

In [ ]:
# ========================================
# 4. ASIGNACIÓN DE SYNSETS DE WORDNET
# ========================================

# Asignación de synsets 
    # Entrada: lista de tuplas con tokens y sus correspondientes etiquetas gramaticales
    # Argumentos: 
         # lesk: Opción de aplicar el algoritmo de Lesk  [Lesk 1986] de la librería wsd para desambiguar
                # Opción por defecto: n (no) --> Se asignará el primer synset del grupo de synsets (el más común)
                # Opción: s (sí) --> Asignar el synset determinado por Lesk
                
def syns(tagged_tokens, desambiguar=None):
    from nltk.wsd import lesk

    syns = list()
    errors = list()
    if desambiguar == True:
        for j in tagged_tokens:
            resultado_lesk = lesk([token for token, tag in tagged_tokens], j[0], j[1])
            if resultado_lesk is not None:
                syns.append(resultado_lesk)
            else:
                errors.append(j)
    else: 
        for j in tagged_tokens:
            try: syns.append(wn.synsets(j[0],j[1])[0])
            except: errors.append(j)
    return(syns, errors)
    
# Ejemplo para comprobar funcionamiento syns()

desambiguar_test = True

print('Desambiguación Lesk:', desambiguar_test )

tagged_tokens = c['tagged_tokens1'][151]
syns1 = syns(tagged_tokens, desambiguar_test )

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

print('****** EJEMPLO syns() ********')   
print('tagged_tokens:', tagged_tokens)
print('syns(tagged_tokens):', syns1[0])
print('syns(tagged_tokens) - Errors:', syns1[1])
print('****')

desambiguar_test = False

print('Desambiguación Lesk:', desambiguar_test )

tagged_tokens = c['tagged_tokens1'][151]
syns1 = syns(tagged_tokens, desambiguar_test )

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

print('****** EJEMPLO syns() ********')   
print('tagged_tokens:', tagged_tokens)
print('syns(tagged_tokens):', syns1[0])
print('syns(tagged_tokens) - Errors:', syns1[1])
print('****')

In [ ]:
# Asignacion de synsets a los tokens de c

desambiguar = desambiguar
print('Desambiguación Lesk:', desambiguar)
tipo_bow = tipo_bow
print('Tipo_bow:', tipo_bow)

c['syns1'] = None
c['syns2'] = None
c['errors1'] = None
c['errors2'] = None
c['syns1_nv'] = None
c['syns2_nv'] = None
c['syns1_resto'] = None
c['syns2_resto'] = None

for i in c.index:

    res1 = syns(c.at[i, 'tagged_tokens1'], desambiguar)
    res2 = syns(c.at[i, 'tagged_tokens2'], desambiguar)

    c.at[i,'syns1'] = res1[0]
    c.at[i,'syns2'] = res2[0]
    c.at[i,'errors1'] = res1[1]
    c.at[i,'errors2'] = res2[1]

    if tipo_bow == 'nv':
        c.at[i,'syns1_nv'] = [syn for syn in res1[0] if syn.pos() in ['n','v']]
        c.at[i,'syns2_nv'] = [syn for syn in res2[0] if syn.pos() in ['n','v']]
        c.at[i,'syns1_resto'] = [syn for syn in res1[0] if syn.pos() not in ['n','v']]
        c.at[i,'syns2_resto'] = [syn for syn in res2[0] if syn.pos() not in ['n','v']]

end_total = time.time()
print('TIEMPO TRANSCURRIDO:', end_total-start_total)

c.sample(5)

###Ç Sentence similarity computation – Overlap similarity

This section computes sentence similarity using an **overlap-based approach**. The similarity between two sentences is estimated by measuring the **degree of overlap between the elements present in both sentences**.

Two variants of the computation are evaluated:

1. **Token overlap** – similarity is computed using the sets of tokens extracted from each sentence. This corresponds to the **original formulation of the method proposed by Agirre et al.**
2. **Synset overlap** – tokens are replaced by their associated **WordNet synsets**, allowing overlap to be computed at the **concept level rather than the surface word level**.

In [ ]:
# 7. SIMILITUD A NIVEL DE FRASE - SOLAPAMIENTO LITERAL (AGIRRE)

# Entrada:
#   - tokens1, tokens2: listas de tokens correspondientes a dos frases
# Salida:
#   - puntuación de similitud entre frases, calculada a partir
#     del número de palabras compartidas por ambas listas
# Método:
#   - Se cuentan las palabras de tokens1 que también aparecen en tokens2
#   - La similitud final se normaliza mediante:
#       2 * alineados / (len(tokens1) + len(tokens2))

def similitud_frase_agirre(tokens1, tokens2):
    len1 = len(tokens1)
    len2 = len(tokens2)
    alineados = len([word for word in tokens1 if word in tokens2])
    similitud = 2 * alineados / (len1 + len2)
    return(similitud)
    
# Ejemplo para comprobar funcionamiento similitud_frase_agirre()

tokens1 = c['tokens1'][501]
tokens2 = c['tokens2'][501]


print('****** EJEMPLO similitud_frase_agirre() ********')   

print(tokens1)
print(tokens2)
print(similitud_frase_agirre(tokens1, tokens2))

In [ ]:
# Cálculo y almacenamiento de las similitudes de Agirre
# Se pueden realizar dos cálculos:
#   a) Con tokens: similitud_frase_agirre(tokens1, tokens2)
#      Esta aproximación corresponde exactamente al método propuesto por Agirre et al.
#   b) Con synsets: similitud_frase_agirre(syns1, syns2)
#      Variante explorada en este trabajo, donde los tokens se sustituyen por sus synsets.

if metodo_frase == 'Agirre':

    c['puntuacion_Agirre'] = 0.0

    for i in c.index:

        start = time.time()

        # --- elegir una de las dos opciones ---
        tokens1 = c.at[i, 'tokens1']
        tokens2 = c.at[i, 'tokens2']
        # tokens1 = c.at[i, 'syns1']   # usar synsets en lugar de tokens
        # tokens2 = c.at[i, 'syns2']

        c.at[i, 'puntuacion_Agirre'] = similitud_frase_agirre(tokens1, tokens2)

        end = time.time()
        print(end-start)

In [ ]:
# Fix export problems with illegal characters in Excel 
import re

ILLEGAL_CHARACTERS_RE = re.compile(r'[\x00-\x08\x0B-\x0C\x0E-\x1F]')

def clean_illegal_chars(value):
    if isinstance(value, str):
        return ILLEGAL_CHARACTERS_RE.sub('', value)
    return value

c_export = c.apply(lambda col: col.map(clean_illegal_chars))

x = datetime.datetime.now()
c_export.to_excel(x.strftime("%y%m%d-%H%M-") + criterios + '.xlsx')